# EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
s1_data_train = pd.read_csv(r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source1.tsv", sep='\t')

In [ ]:
# ============================================================
# Amazon ML Challenge 2026
# Entity Resolution - Source 1 EDA
#
# File:
#   dataset/train/train_source1.tsv
#
# Purpose:
#   Perform comprehensive EDA on Source 1 for:
#   1.  Row count
#   2.  Column count/names
#   3.  Duplicate IDs
#   4.  Missing IDs
#   5.  Missing names
#   6.  Missing addresses
#   7.  Missing countries
#   8.  Country distribution
#   9.  Name unique count
#   10. Name duplicate count
#   11. Name length statistics
#   12. Name token statistics
#   13. Name digit/punctuation/non-ASCII statistics
#   14. Address unique count
#   15. Address duplicate count
#   16. Address length statistics
#   17. Address token statistics
#   18. Address digit/punctuation/non-ASCII statistics
#   19. Normalized-name statistics
#   20. Normalized-address statistics
# ============================================================

import os
import re
import unicodedata
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATA_PATH = r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source1.tsv"
OUTPUT_DIR = r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\student_resource\amazon_er\outputs\eda_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 1. LOAD DATA
# ============================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=True
)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print()


# ============================================================
# 2. BASIC DATASET INFORMATION
# ============================================================

print("=" * 70)
print("1-2. DATASET STRUCTURE")
print("=" * 70)

print(f"Row count    : {len(df):,}")
print(f"Column count : {len(df.columns)}")
print(f"Columns      : {list(df.columns)}")

print("\nData types:")
print(df.dtypes)

print()


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def missing_mask(series):
    """
    Detect standard NaN values, empty strings,
    and whitespace-only strings.
    """
    return series.isna() | series.fillna("").str.strip().eq("")


def safe_string(series):
    """
    Convert a column to strings while preserving missing
    values as empty strings for analysis.
    """
    return series.fillna("").astype(str)


def normalize_text(text):
    """
    Conservative Unicode-aware normalization.

    Steps:
    1. Unicode NFKC normalization
    2. Case folding
    3. Replace punctuation with spaces
    4. Collapse repeated whitespace
    5. Strip leading/trailing whitespace

    Important:
    - Does NOT transliterate languages.
    - Does NOT remove non-ASCII characters.
    - Does NOT perform semantic abbreviation expansion.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode compatibility normalization
    text = unicodedata.normalize("NFKC", text)

    # Better than lower() for Unicode text
    text = text.casefold()

    # Replace punctuation/symbol-like characters with spaces.
    # Keep letters and numbers from all scripts.
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if category.startswith("P"):
            cleaned.append(" ")
        else:
            cleaned.append(char)

    text = "".join(cleaned)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def text_statistics(series):
    """
    Generate character-level statistics for a text column.
    """

    s = safe_string(series)

    # Missing/empty values should not contribute characters
    non_empty = s[s.str.strip() != ""]

    char_lengths = non_empty.str.len()

    token_counts = non_empty.str.split().str.len()

    digit_count = non_empty.apply(
        lambda x: sum(ch.isdigit() for ch in x)
    )

    punctuation_count = non_empty.apply(
        lambda x: sum(
            unicodedata.category(ch).startswith("P")
            for ch in x
        )
    )

    non_ascii_count = non_empty.apply(
        lambda x: sum(not ch.isascii() for ch in x)
    )

    return {
        "non_empty_count": len(non_empty),

        # Length
        "length_min": char_lengths.min(),
        "length_max": char_lengths.max(),
        "length_mean": char_lengths.mean(),
        "length_median": char_lengths.median(),
        "length_std": char_lengths.std(),
        "length_p25": char_lengths.quantile(0.25),
        "length_p75": char_lengths.quantile(0.75),
        "length_p90": char_lengths.quantile(0.90),
        "length_p95": char_lengths.quantile(0.95),
        "length_p99": char_lengths.quantile(0.99),

        # Tokens
        "token_min": token_counts.min(),
        "token_max": token_counts.max(),
        "token_mean": token_counts.mean(),
        "token_median": token_counts.median(),
        "token_std": token_counts.std(),
        "token_p25": token_counts.quantile(0.25),
        "token_p75": token_counts.quantile(0.75),
        "token_p90": token_counts.quantile(0.90),
        "token_p95": token_counts.quantile(0.95),
        "token_p99": token_counts.quantile(0.99),

        # Character properties
        "total_digit_characters": digit_count.sum(),
        "total_punctuation_characters": punctuation_count.sum(),
        "total_non_ascii_characters": non_ascii_count.sum(),

        "records_with_digits": (
            digit_count > 0
        ).sum(),

        "records_with_punctuation": (
            punctuation_count > 0
        ).sum(),

        "records_with_non_ascii": (
            non_ascii_count > 0
        ).sum(),
    }


def duplicate_statistics(series):
    """
    Statistics related to duplicated values.
    """

    s = safe_string(series)

    # Ignore empty/missing values when calculating
    # meaningful duplicate statistics.
    s = s[s.str.strip() != ""]

    value_counts = s.value_counts()

    duplicated_values = value_counts[value_counts > 1]

    return {
        "unique_values": s.nunique(),
        "duplicate_value_count": len(duplicated_values),
        "records_with_duplicate_values": (
            duplicated_values.sum()
        ),
        "maximum_frequency": (
            value_counts.max()
            if len(value_counts) > 0
            else 0
        ),
    }


# ============================================================
# 3-7. MISSING VALUES
# ============================================================

print("=" * 70)
print("3-7. ID AND MISSING VALUE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 3. Duplicate IDs
# ------------------------------------------------------------

duplicate_ids = df["entity_id"].duplicated(keep=False)

print(f"Duplicate entity IDs       : {duplicate_ids.sum():,}")
print(
    f"Unique duplicated ID values : "
    f"{df.loc[duplicate_ids, 'entity_id'].nunique():,}"
)


# ------------------------------------------------------------
# 4. Missing IDs
# ------------------------------------------------------------

id_missing = missing_mask(df["entity_id"])

print(f"Missing entity IDs          : {id_missing.sum():,}")


# ------------------------------------------------------------
# 5-7. Missing fields
# ------------------------------------------------------------

for column in [
    "business_name",
    "business_address",
    "country"
]:
    mask = missing_mask(df[column])

    print(
        f"Missing {column:<17}: "
        f"{mask.sum():,} "
        f"({mask.mean() * 100:.4f}%)"
    )

print()


# Save missing-value summary

missing_summary = []

for column in df.columns:
    mask = missing_mask(df[column])

    missing_summary.append({
        "column": column,
        "missing_count": mask.sum(),
        "missing_percentage": mask.mean() * 100
    })

missing_summary_df = pd.DataFrame(missing_summary)

missing_summary_df.to_csv(
    f"{OUTPUT_DIR}/missing_summary.csv",
    index=False
)


# ============================================================
# 8. COUNTRY DISTRIBUTION
# ============================================================

print("=" * 70)
print("8. COUNTRY DISTRIBUTION")
print("=" * 70)

country_counts = (
    df["country"]
    .fillna("<MISSING>")
    .replace(r"^\s*$", "<EMPTY>", regex=True)
    .value_counts()
)

country_distribution = pd.DataFrame({
    "count": country_counts,
    "percentage": country_counts / len(df) * 100
})

print(country_distribution)

country_distribution.to_csv(
    f"{OUTPUT_DIR}/country_distribution.csv"
)

print()


# ============================================================
# 9-13. BUSINESS NAME ANALYSIS
# ============================================================

print("=" * 70)
print("9-13. BUSINESS NAME ANALYSIS")
print("=" * 70)

name_dup = duplicate_statistics(df["business_name"])

print(f"Unique names                         : {name_dup['unique_values']:,}")
print(f"Duplicated name values               : {name_dup['duplicate_value_count']:,}")
print(
    f"Records belonging to duplicate names : "
    f"{name_dup['records_with_duplicate_values']:,}"
)
print(
    f"Maximum frequency of one name        : "
    f"{name_dup['maximum_frequency']:,}"
)

name_stats = text_statistics(df["business_name"])

print("\nName statistics:")
for key, value in name_stats.items():
    print(f"{key:<35}: {value}")

print()


# ============================================================
# 13. NAME CHARACTER COMPOSITION
# ============================================================

print("-" * 70)
print("NAME CHARACTER COMPOSITION")
print("-" * 70)

names = safe_string(df["business_name"])
non_empty_names = names[names.str.strip() != ""]

name_digit_records = non_empty_names.apply(
    lambda x: any(ch.isdigit() for ch in x)
)

name_punctuation_records = non_empty_names.apply(
    lambda x: any(
        unicodedata.category(ch).startswith("P")
        for ch in x
    )
)

name_non_ascii_records = non_empty_names.apply(
    lambda x: any(not ch.isascii() for ch in x)
)

name_ampersand_records = non_empty_names.str.contains(
    "&",
    regex=False
)

name_slash_records = non_empty_names.str.contains(
    "/",
    regex=False
)

name_hyphen_records = non_empty_names.str.contains(
    "-",
    regex=False
)

name_period_records = non_empty_names.str.contains(
    ".",
    regex=False
)

print(
    f"Names containing digits        : "
    f"{name_digit_records.sum():,} "
    f"({name_digit_records.mean()*100:.2f}%)"
)

print(
    f"Names containing punctuation   : "
    f"{name_punctuation_records.sum():,} "
    f"({name_punctuation_records.mean()*100:.2f}%)"
)

print(
    f"Names containing non-ASCII     : "
    f"{name_non_ascii_records.sum():,} "
    f"({name_non_ascii_records.mean()*100:.2f}%)"
)

print(
    f"Names containing '&'           : "
    f"{name_ampersand_records.sum():,}"
)

print(
    f"Names containing '/'           : "
    f"{name_slash_records.sum():,}"
)

print(
    f"Names containing '-'           : "
    f"{name_hyphen_records.sum():,}"
)

print(
    f"Names containing '.'           : "
    f"{name_period_records.sum():,}"
)

print()


# ============================================================
# 14-18. BUSINESS ADDRESS ANALYSIS
# ============================================================

print("=" * 70)
print("14-18. BUSINESS ADDRESS ANALYSIS")
print("=" * 70)

address_dup = duplicate_statistics(df["business_address"])

print(
    f"Unique addresses                     : "
    f"{address_dup['unique_values']:,}"
)

print(
    f"Duplicated address values             : "
    f"{address_dup['duplicate_value_count']:,}"
)

print(
    f"Records belonging to duplicate addrs : "
    f"{address_dup['records_with_duplicate_values']:,}"
)

print(
    f"Maximum frequency of one address     : "
    f"{address_dup['maximum_frequency']:,}"
)

address_stats = text_statistics(df["business_address"])

print("\nAddress statistics:")
for key, value in address_stats.items():
    print(f"{key:<35}: {value}")

print()


# ============================================================
# 18. ADDRESS CHARACTER COMPOSITION
# ============================================================

print("-" * 70)
print("ADDRESS CHARACTER COMPOSITION")
print("-" * 70)

addresses = safe_string(df["business_address"])
non_empty_addresses = addresses[
    addresses.str.strip() != ""
]

address_digit_records = non_empty_addresses.apply(
    lambda x: any(ch.isdigit() for ch in x)
)

address_punctuation_records = non_empty_addresses.apply(
    lambda x: any(
        unicodedata.category(ch).startswith("P")
        for ch in x
    )
)

address_non_ascii_records = non_empty_addresses.apply(
    lambda x: any(not ch.isascii() for ch in x)
)

print(
    f"Addresses containing digits      : "
    f"{address_digit_records.sum():,} "
    f"({address_digit_records.mean()*100:.2f}%)"
)

print(
    f"Addresses containing punctuation : "
    f"{address_punctuation_records.sum():,} "
    f"({address_punctuation_records.mean()*100:.2f}%)"
)

print(
    f"Addresses containing non-ASCII   : "
    f"{address_non_ascii_records.sum():,} "
    f"({address_non_ascii_records.mean()*100:.2f}%)"
)

print()


# ============================================================
# 19. NORMALIZED NAME ANALYSIS
# ============================================================

print("=" * 70)
print("19. NORMALIZED NAME ANALYSIS")
print("=" * 70)

print("Creating normalized names...")

df["business_name_normalized"] = (
    df["business_name"]
    .apply(normalize_text)
)

normalized_name_dup = duplicate_statistics(
    df["business_name_normalized"]
)

print(
    f"Unique normalized names               : "
    f"{normalized_name_dup['unique_values']:,}"
)

print(
    f"Duplicated normalized name values     : "
    f"{normalized_name_dup['duplicate_value_count']:,}"
)

print(
    f"Records in duplicate normalized names: "
    f"{normalized_name_dup['records_with_duplicate_values']:,}"
)

print(
    f"Maximum normalized-name frequency     : "
    f"{normalized_name_dup['maximum_frequency']:,}"
)

normalized_name_stats = text_statistics(
    df["business_name_normalized"]
)

print("\nNormalized name statistics:")

for key, value in normalized_name_stats.items():
    print(f"{key:<35}: {value}")

print()


# ============================================================
# 20. NORMALIZED ADDRESS ANALYSIS
# ============================================================

print("=" * 70)
print("20. NORMALIZED ADDRESS ANALYSIS")
print("=" * 70)

print("Creating normalized addresses...")

df["business_address_normalized"] = (
    df["business_address"]
    .apply(normalize_text)
)

normalized_address_dup = duplicate_statistics(
    df["business_address_normalized"]
)

print(
    f"Unique normalized addresses               : "
    f"{normalized_address_dup['unique_values']:,}"
)

print(
    f"Duplicated normalized address values     : "
    f"{normalized_address_dup['duplicate_value_count']:,}"
)

print(
    f"Records in duplicate normalized addresses: "
    f"{normalized_address_dup['records_with_duplicate_values']:,}"
)

print(
    f"Maximum normalized-address frequency     : "
    f"{normalized_address_dup['maximum_frequency']:,}"
)

normalized_address_stats = text_statistics(
    df["business_address_normalized"]
)

print("\nNormalized address statistics:")

for key, value in normalized_address_stats.items():
    print(f"{key:<35}: {value}")

print()


# ============================================================
# NORMALIZATION IMPACT
# ============================================================

print("=" * 70)
print("NORMALIZATION IMPACT")
print("=" * 70)

raw_name_unique = (
    safe_string(df["business_name"])
    .replace("", pd.NA)
    .dropna()
    .nunique()
)

normalized_name_unique = (
    df["business_name_normalized"]
    .replace("", pd.NA)
    .dropna()
    .nunique()
)

raw_address_unique = (
    safe_string(df["business_address"])
    .replace("", pd.NA)
    .dropna()
    .nunique()
)

normalized_address_unique = (
    df["business_address_normalized"]
    .replace("", pd.NA)
    .dropna()
    .nunique()
)

print(
    f"Name unique values:"
    f"\n  Raw        : {raw_name_unique:,}"
    f"\n  Normalized : {normalized_name_unique:,}"
    f"\n  Reduction  : "
    f"{raw_name_unique - normalized_name_unique:,}"
)

print(
    f"\nAddress unique values:"
    f"\n  Raw        : {raw_address_unique:,}"
    f"\n  Normalized : {normalized_address_unique:,}"
    f"\n  Reduction  : "
    f"{raw_address_unique - normalized_address_unique:,}"
)

print()


# ============================================================
# SAVE TOP DUPLICATES
# ============================================================

print("=" * 70)
print("SAVING DUPLICATE VALUE REPORTS")
print("=" * 70)

# Raw names
raw_name_counts = (
    safe_string(df["business_name"])
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

raw_name_counts[
    raw_name_counts > 1
].head(100).to_csv(
    f"{OUTPUT_DIR}/top_duplicate_names.csv",
    header=["count"]
)

# Normalized names
normalized_name_counts = (
    df["business_name_normalized"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

normalized_name_counts[
    normalized_name_counts > 1
].head(100).to_csv(
    f"{OUTPUT_DIR}/top_duplicate_normalized_names.csv",
    header=["count"]
)

# Raw addresses
raw_address_counts = (
    safe_string(df["business_address"])
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

raw_address_counts[
    raw_address_counts > 1
].head(100).to_csv(
    f"{OUTPUT_DIR}/top_duplicate_addresses.csv",
    header=["count"]
)

# Normalized addresses
normalized_address_counts = (
    df["business_address_normalized"]
    .replace("", pd.NA)
    .dropna()
    .value_counts()
)

normalized_address_counts[
    normalized_address_counts > 1
].head(100).to_csv(
    f"{OUTPUT_DIR}/top_duplicate_normalized_addresses.csv",
    header=["count"]
)

print(f"Saved reports to: {OUTPUT_DIR}/")
print()


# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 70)
print("SOURCE 1 EDA COMPLETE")
print("=" * 70)

print(f"Rows                         : {len(df):,}")
print(f"Columns                      : {len(df.columns)}")
print(f"Missing IDs                  : {id_missing.sum():,}")
print(f"Missing names                : {missing_mask(df['business_name']).sum():,}")
print(f"Missing addresses            : {missing_mask(df['business_address']).sum():,}")
print(f"Missing countries            : {missing_mask(df['country']).sum():,}")
print(f"Unique raw names             : {raw_name_unique:,}")
print(f"Unique normalized names      : {normalized_name_unique:,}")
print(f"Unique raw addresses         : {raw_address_unique:,}")
print(f"Unique normalized addresses  : {normalized_address_unique:,}")

print("\nOutput files:")
print(f"  {OUTPUT_DIR}/missing_summary.csv")
print(f"  {OUTPUT_DIR}/country_distribution.csv")
print(f"  {OUTPUT_DIR}/top_duplicate_names.csv")
print(f"  {OUTPUT_DIR}/top_duplicate_normalized_names.csv")
print(f"  {OUTPUT_DIR}/top_duplicate_addresses.csv")
print(f"  {OUTPUT_DIR}/top_duplicate_normalized_addresses.csv")

print("\nDone.")